# 05 – Salary model

Goal: predict the yearly salary of a posting (pooled model on all 35,910 LinkedIn salary rows, evaluated separately on data jobs).

1. Feature table (role, level, state, work type, remote, industry, company size, title text, skills).
2. Group train / test split by desc_group.
3. Baselines: global median, median by role + level + state.
4. CatBoost on log(salary).
5. Salary ranges (P10 / P50 / P90) and SHAP explanations.

## Step 1: Feature table

In [1]:
import re
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.skills import extract_skills

RAW = "../data/raw/linkedin"
sal = pd.read_parquet("../data/processed/linkedin_salaries.parquet")
print(sal.shape)

(35910, 39)


In [2]:
# Industry (first one per posting) and latest company size
ind = (pd.read_csv(f"{RAW}/jobs/job_industries.csv")
         .drop_duplicates("job_id")
         .merge(pd.read_csv(f"{RAW}/mappings/industries.csv"), on="industry_id", how="left"))

size = (pd.read_csv(f"{RAW}/companies/employee_counts.csv")
          .sort_values("time_recorded")
          .drop_duplicates("company_id", keep="last")[["company_id", "employee_count"]])

df = (sal.merge(ind[["job_id", "industry_name"]], on="job_id", how="left")
         .merge(size, on="company_id", how="left"))

print("rows:", len(df), "(must still be 35,910)")
print(df[["industry_name", "employee_count"]].isna().mean().round(3))

rows: 35910 (must still be 35,910)
industry_name     0.011
employee_count    0.014
dtype: float64


In [3]:
df["state"] = df["state"].fillna("NO_STATE")
df["remote"] = df["remote_allowed"].fillna(0).astype(int)
df["industry"] = df["industry_name"].fillna("Unknown")
df["log_company_size"] = np.log1p(df["employee_count"])
df["log_salary"] = np.log(df["salary"])

df["skills"] = df["description"].apply(extract_skills)   # ~2–3 minutes

skill_counts = df["skills"].explode().value_counts()
top_skills = skill_counts[skill_counts >= 0.01 * len(df)].index.tolist()
print(len(top_skills), "skills appear in at least 1% of postings")

for s in top_skills:
    df["skill_" + re.sub(r"\W", "_", s)] = df["skills"].apply(lambda x, s=s: s in x).astype(int)

19 skills appear in at least 1% of postings


In [4]:
CAT = ["role", "level", "state", "formatted_work_type", "industry"]
NUM = ["remote", "log_company_size", "is_data_job"]
TEXT = ["title"]
SKILL_COLS = [c for c in df.columns if c.startswith("skill_")]

features = df[["job_id", "desc_group", "salary", "log_salary"] + CAT + NUM + TEXT + SKILL_COLS]
features.to_parquet("../data/processed/salary_features.parquet", index=False)

print(features.shape)
features.head(3)

(35910, 32)


,job_id,desc_group,salary,log_salary,role,level,state,formatted_work_type,industry,remote,...,skill_oracle,skill_linux,skill_r,skill_javascript,skill_power_bi,skill_git,skill_ci_cd,skill_tableau,skill_c__,skill_gcp
0,921716,35d1ac98bd02,38480.0,10.557894,Other,Unknown,NJ,Full-time,Real Estate,0,...,0,0,0,0,0,0,0,0,0,0
1,1829192,211dc20e5f21,83200.0,11.329003,Other,Unknown,CO,Full-time,Unknown,0,...,0,0,0,0,0,0,0,0,0,0
2,10998357,feeadef93690,55000.0,10.915088,Other,Lead/Principal,OH,Full-time,Restaurants,0,...,0,0,0,0,0,0,0,0,0,0


### Feature table

- 35,910 rows (no duplicates from the merges), 32 columns: role (only for data / AI postings, else Other), is_data_job, level, state (NO_STATE for 17%), work type, industry (338 values, 1.1% unknown), log company size (1.4% missing), remote, title as text, 19 skill flags (skills in ≥ 1% of postings).
- Bug found and fixed: role_family was applied to all titles, so 6,166 non-data jobs got a data role (Software Engineer → Data Engineer, Financial Analyst → Data Analyst, Bench Scientist → Data Scientist). Now the role is only set for postings that passed the data / AI title filter in notebook 01.
- Excluded on purpose: min / max / med salary, raw_mid, pay_period, compensation_type (built from the salary or only filled when a salary exists), views / applies / closed_time (known only after posting), company_name (phase 2).
- Known gap: the data / AI title filter misses 10 "GenAI" titles (no word boundary inside "GenAI").

## Step 2: Group train / test split

In [5]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(features, groups=features["desc_group"]))
train = features.iloc[train_idx].copy()
test = features.iloc[test_idx].copy()

print(f"train: {len(train)} rows | test: {len(test)} rows ({len(test) / len(features):.1%})")
print("descriptions in both train and test:", len(set(train["desc_group"]) & set(test["desc_group"])))

for name, d in [("train", train), ("test", test)]:
    print(f"{name}: data jobs {d['is_data_job'].sum()} | median salary {d['salary'].median():,.0f}")

train: 28705 rows | test: 7205 rows (20.1%)
descriptions in both train and test: 0
train: data jobs 752 | median salary 82,500
test: data jobs 206 | median salary 82,500


In [6]:
# What a normal random split would have done
r_train, r_test = train_test_split(features, test_size=0.2, random_state=42)
leaked = r_test["desc_group"].isin(r_train["desc_group"]).sum()
print(f"random split: {leaked} test rows ({leaked / len(r_test):.1%}) have a copy of their description in train")

random split: 556 test rows (7.7%) have a copy of their description in train


### Group split

- GroupShuffleSplit on desc_group: 28,705 train / 7,205 test rows (20.1%), 0 descriptions in both, same median salary (82.5k).
- Data jobs: 752 train / 206 test (small test set → data-job metrics are noisy).
- A normal random split would have put copies of the same description in train for 556 test rows (7.7%) → leakage.

## Step 3: Baselines

In [7]:
def evaluate(y_true_log, y_pred_log, name):
    """Metrics in dollars (after exp) and R² on the log scale."""
    y_true_log, y_pred_log = np.asarray(y_true_log), np.asarray(y_pred_log)
    yt, yp = np.exp(y_true_log), np.exp(y_pred_log)
    abs_err = np.abs(yp - yt)
    r2 = 1 - ((y_true_log - y_pred_log) ** 2).sum() / ((y_true_log - y_true_log.mean()) ** 2).sum()
    return {"model": name, "MAE $": round(abs_err.mean()), "median AE $": round(np.median(abs_err)),
            "MAPE %": round((abs_err / yt).mean() * 100, 1), "R2 log": round(r2, 3)}


def group_median_predict(train, test, key_chain, min_count=5):
    """Median log salary of the most specific group with at least min_count train rows."""
    pred = pd.Series(np.nan, index=test.index)
    for keys in key_chain:
        g = train.groupby(keys)["log_salary"]
        med = g.median()[g.size() >= min_count].rename("group_median")
        pred = pred.fillna(test[keys].join(med, on=keys)["group_median"])
    return pred.fillna(train["log_salary"].median())

In [8]:
is_dj = test["is_data_job"] == 1
preds = {
    "global median": np.full(len(test), train["log_salary"].median()),
    "group median (role + level + state)": group_median_predict(
        train, test, [["role", "level", "state"], ["role", "level"], ["role"]]).to_numpy(),
}

rows = []
for name, p in preds.items():
    rows.append({"subset": "all test", **evaluate(test["log_salary"], p, name)})
    rows.append({"subset": "data jobs", **evaluate(test.loc[is_dj, "log_salary"], p[is_dj.to_numpy()], name)})

results = pd.DataFrame(rows)
results

,subset,model,MAE $,median AE $,MAPE %,R2 log
0,all test,global median,42547,34208,50.8,-0.000
1,data jobs,global median,69721,53088,42.2,-1.650
2,all test,group median (role + level + state),34078,23220,39.2,0.290
3,data jobs,group median (role + level + state),37372,26075,26.6,0.169


### Baseline results

| subset | model | MAE $ | median AE $ | MAPE % | R² log |
|---|---|---|---|---|---|
| all test | global median | 42,547 | 34,208 | 50.8 | 0.000 |
| data jobs | global median | 69,721 | 53,088 | 42.2 | -1.650 |
| all test | group median (role + level + state) | 34,078 | 23,220 | 39.2 | 0.290 |
| data jobs | group median (role + level + state) | 37,372 | 26,075 | 26.6 | 0.169 |

- Negative R² for data jobs with the global median: predicting 82.5k for postings with a median of ~137k is worse than predicting their own mean.
- Group median (with fallback role + level → role → global, min 5 rows per group) is the bar CatBoost has to beat: median error 26k for data jobs, R² only 0.17 → role, level and state explain little of the salary differences inside data jobs.